# 连续词袋模型

## 第一阶段：项目准备与数据获取
首先，我们需要导入必要的库。

In [ ]:
# ============ 导入必要的库 ============
import torch  # PyTorch框架，主要用于深度学习
import torch.nn as nn  # 神经网络模块，包含各种层的定义
import torch.optim as optim  # 优化器模块，用于训练模型
from torch.utils.data import Dataset, DataLoader  # 数据加载工具，便于批量处理数据
import numpy as np  # 数值计算库，用于向量运算（如余弦相似度）
import re  # 正则表达式库，用于文本清洗
import requests  # HTTP请求库，用于下载文本数据
from collections import Counter  # 计数器，用于统计词频

# ============ 1. 配置参数 (Hyperparameters) ============
# 超参数是机器学习中人为设置的参数，不是从数据中学习的
class Config:
    """
    存储所有配置参数的类，集中管理超参数便于后续调整
    """
    EMBED_DIM = 100         # 词向量的维度
                            # 例如：每个词用100维向量表示
                            # 维度越高，表示能力越强，但计算量也越大
    
    CONTEXT_SIZE = 2        # 上下文窗口大小（左右各取2个词）
                            # 例如：文本"the quick brown fox"，
                            # 当前词是"quick"时，上下文是["the", "brown", "fox"]
                            # （注意不包括"quick"本身）
    
    BATCH_SIZE = 128        # 批次大小：每次处理128个样本
                            # 不能处理太小防止GPU闲置，太大会导致内存溢出
    
    LEARNING_RATE = 0.001   # 学习率：控制参数更新的步长
                            # 太大容易震荡，太小收敛太慢
    
    EPOCHS = 10             # 训练轮数：完整遍历一遍数据集10次
                            # 每遍一次数据集，模型的参数会更新多次
    
    MIN_FREQ = 2            # 最小词频：出现次数少于2次的词会被过滤掉
                            # 这样可以减小词表大小，只保留重要的词

config = Config()

# 选择计算设备（GPU or CPU）
# GPU速度快，但需要NVIDIA显卡；CPU兼容性好但速度慢
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 第二阶段：数据清洗与词表构建 (Preprocessing)
原始文本是无法直接喂给神经网络的。我们需要做三件事：
1. 清洗：去除标点、转小写。
2. 构建词表：给每个词分配唯一的 ID。
3. 生成训练对：(Context, Target)。

In [ ]:
# ============ 2. 获取并清洗数据 ============

def get_alice_text():
    """
    从古腾堡计划获取《爱丽丝梦游仙境》的英文文本
    
    返回:
        tokens: 清洗后的词标记列表，包含所有单词
        
    处理步骤：
    1. 下载原始文本
    2. 去除古腾堡头尾声明，只保留正文
    3. 转换为小写
    4. 用正则表达式提取单词
    """
    # 古腾堡计划的URL，提供免费电子书
    url = "https://www.gutenberg.org/files/11/11-0.txt"
    
    # 发送HTTP请求下载文本
    response = requests.get(url)
    text = response.text
    
    # ========== 去除古腾堡项目的头尾声明 ==========
    # 古腾堡的文件开头和结尾有版权声明，需要去除，只保留实际的正文
    start = text.find("*** START OF THE PROJECT GUTENBERG EBOOK")
    end = text.find("*** END OF THE PROJECT GUTENBERG EBOOK")
    
    # 如果找到了这两个标记，就提取中间部分
    if start != -1 and end != -1:
        text = text[start:end]
    
    # ========== 文本清洗 ==========
    # 转换为小写，便于后续处理（不区分大小写）
    text = text.lower()
    
    # 使用正则表达式提取所有英文单词
    # r'\b[a-z]+\b' 的含义：
    #   \b: 单词边界
    #   [a-z]+: 一个或多个小写字母
    # 这样做的好处：自动过滤掉了数字和特殊符号
    tokens = re.findall(r'\b[a-z]+\b', text)
    
    return tokens

# ========== 下载和处理文本 ==========
print("Downloading and processing text...")
raw_tokens = get_alice_text()
print(f"Total tokens: {len(raw_tokens)}")  # 打印词标记总数

# ============ 3. 构建词汇表 (Vocabulary) ============

# 统计每个词出现的次数
# Counter 是一个特殊的字典，可以自动计数
vocab_counter = Counter(raw_tokens)

# 初始化词汇表字典
# 格式：词 -> 索引（整数），用于将词转换为数字
# <UNK>（Unknown）用于表示不在词表中的词
vocab = {"<UNK>": 0}

# 反向词汇表：索引 -> 词，用于将数字转换回词
idx_to_word = {0: "<UNK>"}

# 构建词汇表
# 只有出现频次 >= MIN_FREQ 的词才会被加入词表
for word, count in vocab_counter.items():
    if count >= config.MIN_FREQ:
        idx = len(vocab)  # 新词的索引就是当前词表的大小
        vocab[word] = idx
        idx_to_word[idx] = word

VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE}")  # 打印最终的词表大小

# ========== 将文本转换为索引列表 ==========
# 每个词都被转换为它在词表中的索引
# 如果词不在词表中，则使用 <UNK> 的索引（0）
encoded_text = [vocab.get(token, vocab["<UNK>"]) for token in raw_tokens]
# 例如："the quick brown" -> [5, 123, 456] （假设这些是对应的索引）

Total tokens: 27181
Vocabulary size: 1465


## 第三阶段：构建 PyTorch Dataset (Dataset Pipeline)
PyTorch 的核心优势在于 Dataset 和 DataLoader。这能帮我们自动处理 Batch（批次）和 Shuffle（打乱），这是工业级训练的基础。
CBOW 的数据格式是：
* 输入 (X): 上下文单词的索引列表 $[w_{t-2}, w_{t-1}, w_{t+1}, w_{t+2}]$
* 标签 (Y): 中心词的索引 $w_t$

In [ ]:
# ============ 构建 PyTorch Dataset (Dataset Pipeline) ============

class CBOWDataset(Dataset):
    """
    CBOW（连续词袋）模型的数据集类
    
    核心思想：给定上下文词，预测中心词
    例如：文本"the quick brown fox jumps"
    当 context_size=2 时：
    - ("the", "brown", "fox") -> "quick"  （上文"the"，下文"brown", "fox"，预测"quick"）
    - ("quick", "fox", "jumps") -> "brown"
    
    CBOW与Skip-gram的区别：
    - CBOW: 上下文 -> 中心词 (多对一)
    - Skip-gram: 中心词 -> 上下文 (一对多)
    """
    
    def __init__(self, encoded_text, context_size):
        """
        初始化数据集
        
        参数:
            encoded_text: 已编码的文本列表，包含词的索引
                         例如：[5, 123, 456, 789, ...]
            context_size: 上下文窗口大小
                         例如：context_size=2 表示左右各取2个词
        """
        self.data = []
        
        # ========== 滑动窗口生成训练数据 ==========
        # 从第 context_size 个词开始，到倒数第 context_size 个词结束
        # 这样确保每个中心词都有完整的上下文
        for i in range(context_size, len(encoded_text) - context_size):
            # 当前位置的词是目标词（要预测的词）
            target = encoded_text[i]
            
            # ========== 获取上下文 ==========
            # 上下文包括：
            # 1. 左边 context_size 个词：encoded_text[i - context_size : i]
            # 2. 右边 context_size 个词：encoded_text[i + 1 : i + context_size + 1]
            # （注意：中心词本身不包括在内）
            
            context = (
                encoded_text[i - context_size : i] +      # 左上下文
                encoded_text[i + 1 : i + context_size + 1] # 右上下文
            )
            
            # 将 (上下文, 目标词) 对添加到数据列表
            self.data.append((context, target))
            
    def __len__(self):
        """
        返回数据集的大小（样本数量）
        DataLoader 会调用这个方法确定要迭代多少次
        """
        return len(self.data)
    
    def __getitem__(self, idx):
        """
        获取指定索引的样本
        
        参数:
            idx: 样本索引（0到len(dataset)-1）
        
        返回:
            context: 上下文词的索引张量，形状 (context_window_size,)
            target: 目标词的索引张量，标量
        
        注意：必须转换为 Tensor，因为 PyTorch DataLoader 需要张量
        """
        context, target = self.data[idx]
        # 将 Python 列表转换为 PyTorch 张量
        return torch.tensor(context), torch.tensor(target)

# ========== 实例化 Dataset 和 DataLoader ==========

# 创建数据集对象
dataset = CBOWDataset(encoded_text, config.CONTEXT_SIZE)

# 创建数据加载器
# shuffle=True 表示打乱数据顺序，有利于训练（防止有序性导致过拟合）
dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)

# ========== 检查数据集 ==========
print(f"Number of training pairs: {len(dataset)}")  # 打印训练对的数量

# 检查一个样本，确保数据格式正确
sample_context, sample_target = dataset[0]
print(f"Sample Context (indices): {sample_context}")  # 上下文词的索引
print(f"Sample Target (index): {sample_target}")      # 目标词的索引

Number of training pairs: 27177
Sample Context (indices): tensor([0, 1, 0, 0])
Sample Target (index): 2


## 第四阶段：搭建模型 (Model Architecture)
这是最核心的部分。Embedding的实现见`词嵌入-跳元模型-demo`。

In [ ]:
# ============ 搭建模型 (Model Architecture) ============

class CBOW(nn.Module):
    """
    连续词袋 (Continuous Bag of Words) 模型
    
    模型架构：
    1. Embedding 层：将词索引转换为词向量
       输入维度：词表大小（可能的词索引）
       输出维度：词嵌入维度（每个词用多少维向量表示）
    
    2. 聚合层：将上下文词的向量求平均
       这样每个上下文词的贡献是等量的
    
    3. 线性层：从隐藏表示映射回词表空间
       输入维度：词嵌入维度
       输出维度：词表大小（预测词表中哪个词）
    
    数据流：
    词索引 -> 嵌入向量 -> 求平均 -> 线性变换 -> 词表上的分数（logits）
    """
    
    def __init__(self, vocab_size, embed_dim):
        """
        初始化 CBOW 模型
        
        参数:
            vocab_size: 词汇表的大小（不同词的总数）
            embed_dim: 词嵌入的维度（每个词用多少维向量表示）
        """
        super(CBOW, self).__init__()
        
        # ========== 1. 词嵌入层 ==========
        # nn.Embedding(num_embeddings, embedding_dim)
        # 作用：根据词索引查找对应的词向量
        # 例如：如果 vocab_size=10000, embed_dim=100
        #       就相当于一个 10000x100 的矩阵，行索引是词的索引，每行是一个100维向量
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        
        # ========== 2. 线性层（输出层） ==========
        # nn.Linear(in_features, out_features)
        # 作用：将隐藏层（embed_dim 维）映射到词表大小
        # 目的：预测出现每个词的概率（在交叉熵损失前）
        self.linear = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, inputs):
        """
        前向传播：输入 -> 输出
        
        参数:
            inputs: 形状为 (batch_size, context_window_size)
                   例如：batch_size=128, context_window_size=4（左右各2个词）
                   contains 词索引，范围 [0, vocab_size)
        
        返回:
            output: 形状为 (batch_size, vocab_size)
                   每个位置是该词的分数（logits），越高表示模型认为概率越高
        """
        
        # ========== 步骤1：获取嵌入向量 ==========
        # self.embeddings(inputs) 根据索引查找词向量
        embeds = self.embeddings(inputs)
        # embeds 的形状：(batch_size, context_window_size, embed_dim)
        # 例如：(128, 4, 100) 表示 128 个样本，每个 4 个上下文词，每个词 100 维向量
        
        # ========== 步骤2：聚合上下文 (CBOW 的核心) ==========
        # torch.mean(embeds, dim=1) 在 context_window_size 这个维度求平均
        # 这样将多个词向量聚合成一个向量，作为整个上下文的表示
        hidden = torch.mean(embeds, dim=1)
        # hidden 的形状：(batch_size, embed_dim)
        # 例如：(128, 100) 表示 128 个样本，每个生成一个 100 维的上下文向量
        
        # 为什么求平均？
        # - 上下文词的贡献是等量的
        # - 此外还可以选择 torch.sum()（求和）或其他聚合方式
        
        # ========== 步骤3：预测层 ==========
        # self.linear(hidden) 将隐藏向量映射到词表空间
        output = self.linear(hidden)
        # output 的形状：(batch_size, vocab_size)
        # 例如：(128, 10000) 表示 128 个样本，每个对词表中 10000 个词的预测分数
        
        return output

# ========== 创建模型实例 ==========
# 将模型移到指定设备（GPU 或 CPU）
model = CBOW(VOCAB_SIZE, config.EMBED_DIM).to(device)

# 打印模型结构，查看参数数量
print(model)

CBOW(
  (embeddings): Embedding(1465, 100)
  (linear): Linear(in_features=100, out_features=1465, bias=True)
)


## 第五阶段：训练循环 (Training Loop)
这里我们使用 CrossEntropyLoss。

注意：nn.CrossEntropyLoss 在内部已经包含了 LogSoftmax 和 NLLLoss，所以我们在模型的输出层不需要手动加 Softmax。

In [ ]:
# ============ 训练循环 (Training Loop) ============

# ========== 创建优化器 ==========
# Adam 优化器是目前最流行的优化算法之一
# 参数说明：
#   model.parameters()：要优化的参数（模型的所有权重和偏置）
#   lr：学习率，控制每次更新的步长大小
optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)

# ========== 创建损失函数 ==========
# CrossEntropyLoss：交叉熵损失函数，用于多分类问题
# 为什么适合这里？
# - 模型输出的是 vocab_size 个词的分数（logits）
# - 我们要选出正确的词
# 这是一个多分类问题，每个样本有 vocab_size 个类别
loss_function = nn.CrossEntropyLoss()

# 注意：nn.CrossEntropyLoss 在内部已经包含了 LogSoftmax 和 NLLLoss
# 所以模型的输出层不需要手动加 Softmax 或 Log

# ========== 开始训练 ==========
print("Start Training...")

for epoch in range(config.EPOCHS):
    """
    epoch：一个完整的数据集遍历
    每个 epoch 会：
    1. 遍历数据集中的所有批次
    2. 对每个批次进行前向传播、计算损失、反向传播、更新参数
    3. 累计损失并计算平均损失
    """
    
    total_loss = 0  # 累计本 epoch 的总损失
    
    # ========== 遍历每个批次 ==========
    for context, target in dataloader:
        """
        dataloader 会自动：
        1. 从 dataset 中取出一个批次的样本
        2. 将它们堆叠成张量
        3. 支持 shuffle（打乱顺序）和并行加载
        """
        
        # ========== 1. 数据移到设备 ==========
        # 确保数据和模型在同一设备上（GPU 或 CPU）
        # 否则会报错
        context = context.to(device)
        target = target.to(device)
        
        # ========== 2. 梯度清零 ==========
        # PyTorch 中，梯度是累加的（而不是覆盖）
        # 所以每次反向传播前必须先清零上次的梯度
        model.zero_grad()
        
        # ========== 3. 前向传播 ==========
        # 将数据输入模型，得到预测分数
        log_probs = model(context)
        # log_probs 的形状：(batch_size, vocab_size)
        # 每个值表示对应词的预测分数（logits）
        
        # ========== 4. 计算损失 ==========
        # CrossEntropyLoss 会计算模型的预测与真实标签的差异
        loss = loss_function(log_probs, target)
        # loss 是一个标量，范围通常是 [0, ∞)
        # 损失越小，模型预测越准确
        
        # ========== 5. 反向传播 ==========
        # loss.backward() 计算每个参数的梯度
        # 梯度表示该参数应该如何改变来减小损失
        loss.backward()
        # 反向传播过程：
        # 损失 -> 线性层梯度 -> 嵌入层梯度 -> ... -> 第一层梯度
        
        # ========== 6. 更新参数 ==========
        # optimizer.step() 根据梯度更新参数
        # 新参数 = 旧参数 - 学习率 * 梯度
        optimizer.step()
        # 这样模型会逐渐向更好的方向调整
        
        # ========== 7. 累计损失 ==========
        # .item() 将单元素张量转换为 Python 标量
        total_loss += loss.item()
    
    # ========== 每个 epoch 结束后 ==========
    # 计算并打印平均损失
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{config.EPOCHS}, Loss: {avg_loss:.4f}")
    # 正常情况下，损失应该逐渐下降，说明模型在学习

print("Training Finished!")

Start Training...
Epoch 1/10, Loss: 6.7122
Epoch 2/10, Loss: 5.6935
Epoch 3/10, Loss: 5.2916
Epoch 4/10, Loss: 5.0184
Epoch 5/10, Loss: 4.7951
Epoch 6/10, Loss: 4.6002
Epoch 7/10, Loss: 4.4319
Epoch 8/10, Loss: 4.2801
Epoch 9/10, Loss: 4.1454
Epoch 10/10, Loss: 4.0188
Training Finished!


## 第六阶段：模型应用与可视化
训练完了，怎么知道模型好不好？我们写一个函数，输入一个词，找出和它最相似的词。这通过计算余弦相似度 (Cosine Similarity) 来实现。

In [ ]:
def get_similar_words(word, n=5):
    """
    给定一个词，找出与它最相似的 n 个词
    
    原理：
    1. 获取输入词的词向量
    2. 计算它与所有其他词向量的余弦相似度
    3. 返回相似度最高的 n 个词
    
    为什么使用余弦相似度？
    - 余弦相似度衡量两个向量方向的相似程度
    - 范围：[-1, 1]，1 表示完全相同方向，-1 表示完全相反，0 表示正交
    - 对于词向量，方向比大小更重要
    
    参数:
        word: 查询的词（字符串）
        n: 返回最相似的词的个数，默认为 5
    """
    
    # ========== 1. 检查词是否在词表中 ==========
    if word not in vocab:
        print(f"Word '{word}' not in vocabulary.")
        return

    # ========== 2. 获取输入词的向量 ==========
    # 根据词查找其索引
    word_idx = vocab[word]
    
    # 从嵌入层的权重矩阵中提取该词的向量
    # model.embeddings.weight 是嵌入矩阵，形状 (vocab_size, embed_dim)
    # .cpu() 确保在 CPU 上进行计算（即使模型在 GPU）
    # .detach() 分离计算图，因为我们只需要向量值，不需要梯度
    # .numpy() 转换为 numpy 数组，便于进行数值计算
    word_vec = model.embeddings.weight[word_idx].cpu().detach().numpy()
    # word_vec 的形状：(embed_dim,)，例如 (100,)
    
    # ========== 3. 获取所有词的向量矩阵 ==========
    # model.embeddings.weight 整个嵌入矩阵
    all_weights = model.embeddings.weight.cpu().detach().numpy()
    # all_weights 的形状：(vocab_size, embed_dim)，例如 (5000, 100)
    
    # ========== 4. 计算余弦相似度 ==========
    # 余弦相似度公式：Cosine Sim = (A · B) / (|A| * |B|)
    # 其中：
    #   A · B 是向量的点积（内积）
    #   |A| 和 |B| 是向量的范数（长度）
    
    # 计算分子：点积
    # np.dot(all_weights, word_vec) 计算每个词向量与输入词向量的点积
    # 形状：(vocab_size,)
    dot_product = np.dot(all_weights, word_vec)
    
    # 计算分母：所有向量的范数
    # np.linalg.norm(all_weights, axis=1) 计算每个词向量的欧几里得范数
    # axis=1 表示沿词向量维度计算（所以结果是每个词一个范数）
    # 形状：(vocab_size,)
    norm_all = np.linalg.norm(all_weights, axis=1)
    
    # 计算分母：输入词向量的范数
    norm_word = np.linalg.norm(word_vec)
    # norm_word 是一个标量
    
    # 计算余弦相似度
    # 广播机制：norm_all 是 (vocab_size,)，norm_word 是标量
    # 结果是 (vocab_size,)，每个元素是对应词的余弦相似度
    similarities = dot_product / (norm_all * norm_word)
    
    # ========== 5. 排序找出最相似的词 ==========
    # np.argsort(similarities) 返回排序后的索引
    # 例如：如果 similarities = [0.5, 0.8, 0.3]，
    #       argsort 返回 [2, 0, 1]（从小到大的索引顺序）
    # [-n-1:-1] 选择最后 n 个最大的索引（最相似的词），除了最后一个（通常是查询词本身）
    # [::-1] 反转顺序，从大到小排列
    top_indices = np.argsort(similarities)[-n-1:-1][::-1]
    
    # ========== 6. 打印结果 ==========
    print(f"\nWords closest to '{word}':")
    
    # 遍历最相似的词的索引
    for idx in top_indices:
        # 将索引转换回词
        word_name = idx_to_word[idx]
        # 获取相似度分数
        similarity_score = similarities[idx]
        # 打印词和对应的相似度分数
        print(f"  {word_name} (Sim: {similarity_score:.4f})")

# ========== 测试模型 ==========
# 搜索与《爱丽丝梦游仙境》中典型词汇最相似的词
# 这验证了模型是否学到了词的语义关系

get_similar_words("alice")   # "alice"（爱丽丝）应该与人物名相似
get_similar_words("queen")   # "queen"（皇后）应该与角色词相似
get_similar_words("rabbit")  # "rabbit"（兔子）应该与动物词相似
get_similar_words("tea")     # "tea"（茶）应该与饮食词相似


Words closest to 'alice':
  exactly (Sim: 0.3279)
  rome (Sim: 0.3033)
  stop (Sim: 0.3032)
  done (Sim: 0.2981)
  listen (Sim: 0.2859)

Words closest to 'queen':
  sends (Sim: 0.3515)
  mouse (Sim: 0.3470)
  dish (Sim: 0.3455)
  only (Sim: 0.3246)
  pie (Sim: 0.3211)

Words closest to 'rabbit':
  felt (Sim: 0.3748)
  blow (Sim: 0.3481)
  executioner (Sim: 0.3355)
  fashion (Sim: 0.3210)
  staring (Sim: 0.3203)

Words closest to 'tea':
  will (Sim: 0.2942)
  dance (Sim: 0.2777)
  box (Sim: 0.2775)
  man (Sim: 0.2764)
  loudly (Sim: 0.2742)


: 